# تدريب نموذج تشخيص سرطان الجلد المبكر — HAM10000 (نسخة محفوظة على Drive)
مشروع تخرج: سلمى محمد عيسى | إشراف: د. محمد حجوز

**الفرق عن النسخة القديمة:** هذه النسخة تحفظ البيانات ونقاط التقدم (checkpoints) على Google Drive،
فإذا انقطع الاتصال أو أُغلقت الجلسة، **ما رح تحتاجي تعيدي تحميل البيانات من جديد**،
وممكن تكملي التدريب من آخر نقطة محفوظة بدل ما تبلشي من الصفر.

**قبل البدء:**
1. Runtime → Change runtime type → اختاري **GPU (T4)**.
2. جهّزي ملف `kaggle.json` (نفس اللي استخدمتيه المرة الماضية).
3. شغّلي الخلايا بالترتيب من فوق لتحت **بدون تخطي أي خلية**.

## 1) ربط Google Drive
بتطلعلك نافذة صلاحية — وافقي عليها. كل البيانات والنموذج رح تُحفظ داخل مجلد `MyDrive/skin_cancer_project/` بحسابك.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/skin_cancer_project"
DATA_DIR = f"{PROJECT_DIR}/ham10000_data"
CHECKPOINT_PATH = f"{PROJECT_DIR}/best_model.h5"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("مجلد المشروع على Drive:", PROJECT_DIR)

## 2) تثبيت المكتبات

In [ ]:
!pip install -q kaggle tensorflow opencv-python-headless pandas numpy matplotlib scikit-learn

## 3) إعداد مصادقة Kaggle
ارفعي ملف `kaggle.json`. بيتحفظ هالمرة داخل بيئة الجلسة (مش على Drive لأسباب أمان — بيانات الدخول ما لازم تُخزَّن بمكان دائم مشترك).

In [ ]:
from google.colab import files
uploaded = files.upload()  # اختاري kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('تم إعداد مفتاح Kaggle بنجاح')

## 4) تحميل بيانات HAM10000 (مرة واحدة فقط)
هذه الخلية **ذكية**: تتحقق أولاً إذا البيانات موجودة على Drive من محاولة سابقة، وإذا كانت موجودة تتخطى التحميل بالكامل وتوفّر عليك الوقت.

In [ ]:
import os

metadata_check = f"{DATA_DIR}/HAM10000_metadata.csv"

if os.path.exists(metadata_check):
    print("✅ البيانات موجودة مسبقاً على Drive - تم تخطي التحميل")
else:
    print("⏳ لم يتم العثور على البيانات، جاري التحميل من Kaggle...")
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content
    !unzip -q -o /content/skin-cancer-mnist-ham10000.zip -d {DATA_DIR}  # -o = استبدال تلقائي بدون سؤال
    print("✅ تم التحميل والحفظ على Drive بنجاح")

!ls {DATA_DIR}

## 5) دمج مجلدي الصور (مرة واحدة فقط - نفس منطق التخطي)

In [ ]:
import os, shutil

IMAGES_DIR = f"{DATA_DIR}/all_images"

if os.path.exists(IMAGES_DIR) and len(os.listdir(IMAGES_DIR)) > 9000:
    print("✅ الصور مدموجة مسبقاً -", len(os.listdir(IMAGES_DIR)), "صورة")
else:
    os.makedirs(IMAGES_DIR, exist_ok=True)
    for part in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        part_path = f"{DATA_DIR}/{part}"
        if os.path.exists(part_path):
            for fname in os.listdir(part_path):
                shutil.copy(os.path.join(part_path, fname), IMAGES_DIR)
    print("✅ تم الدمج - عدد الصور:", len(os.listdir(IMAGES_DIR)))

## 6) استكشاف البيانات

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(metadata_check)
print("عدد الصور الإجمالي:", len(df))
print("\nتوزيع الفئات:")
print(df["dx"].value_counts())

df["dx"].value_counts().plot(kind="bar", figsize=(8,5), color="#028090")
plt.title("توزيع فئات الآفات الجلدية")
plt.tight_layout()
plt.show()

## 7) تقسيم البيانات (Train / Validation)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df["dx"], random_state=42
)
print("عدد صور التدريب:", len(train_df))
print("عدد صور التحقق:", len(val_df))

## 8) مولّدات البيانات (Preprocessing + Augmentation)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_df = train_df.copy(); val_df = val_df.copy()
train_df["filename"] = train_df["image_id"] + ".jpg"
val_df["filename"] = val_df["image_id"] + ".jpg"

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0, rotation_range=30, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True, vertical_flip=True,
    zoom_range=0.15, brightness_range=[0.85, 1.15], fill_mode="nearest",
)
val_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df, directory=IMAGES_DIR, x_col="filename", y_col="dx",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical",
)
val_gen = val_datagen.flow_from_dataframe(
    dataframe=val_df, directory=IMAGES_DIR, x_col="filename", y_col="dx",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False,
)
print("الفئات:", train_gen.class_indices)

## 9) حساب أوزان الفئات (لمعالجة عدم التوازن)

In [ ]:
from sklearn.utils import class_weight
import numpy as np

classes = np.unique(train_df["dx"])
weights = class_weight.compute_class_weight(
    class_weight="balanced", classes=classes, y=train_df["dx"]
)
class_indices = train_gen.class_indices
class_weights = {class_indices[c]: w for c, w in zip(classes, weights)}
print("أوزان الفئات:", class_weights)

> ⛔ **تنبيه:** إذا قرّرتي تستخدمي معمارية EfficientNetB0 (قسم 15 بآخر الدفتر)،
> **لا تشغّلي أي خلية من هون لغاية آخر قسم 14**. انتقلي مباشرة لقسم 15.
> تشغيل هالخلايا فوق موديل محمّل مسبقاً من مرحلة Fine-Tuning سابقة بيسبب خطأ
> `ValueError: Unknown variable...` بسبب تعارض حالة الـ optimizer مع الطبقات.

## 10) بناء النموذج (MobileNetV2) — أو استكمال من آخر نقطة محفوظة
هذه الخلية تتحقق أولاً إذا في نموذج محفوظ سابقاً على Drive من محاولة تدريب سابقة انقطعت، وتكمّل منه بدل البدء من الصفر.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam

if os.path.exists(CHECKPOINT_PATH):
    print("✅ تم العثور على نموذج محفوظ سابقاً - جاري الاستكمال منه")
    model = load_model(CHECKPOINT_PATH)
else:
    print("⏳ لا يوجد نموذج محفوظ - بناء نموذج جديد من الصفر")
    base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
    base_model.trainable = False
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(len(class_indices), activation="softmax")(x)
    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()

## 11) التدريب (يُحفظ تلقائياً على Drive بعد كل تحسّن)
**هذه أهم خلية:** كل ما تتحسن دقة النموذج، بينحفظ فوراً على Drive بملف `best_model.h5`.
لو انقطع الاتصال، فقط أعيدي تشغيل الدفتر من الخلية الأولى (ربط Drive) ثم مباشرة لهذه الخلية —
كل الخلايا فوق ذكية وبتتخطى الخطوات المُنجَزة، وهذه الخلية رح تكمّل من آخر نموذج محفوظ.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# إعادة تجميع أمان لتفادي تعارض optimizer مع حالة الطبقات الحالية
model.compile(optimizer=Adam(learning_rate=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    ModelCheckpoint(CHECKPOINT_PATH, monitor="val_accuracy", save_best_only=True),
]

history = model.fit(
    train_gen, validation_data=val_gen, epochs=15,
    class_weight=class_weights, callbacks=callbacks,
)

## 12) تقييم النموذج (Accuracy, Precision, Recall, F1)

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

val_gen.reset()
preds = model.predict(val_gen)
y_pred = np.argmax(preds, axis=1)
y_true = val_gen.classes

class_names = list(class_indices.keys())
print(classification_report(y_true, y_pred, target_names=class_names))

## 12.5) جولة تحسين إضافية (Fine-Tuning)
نفك تجميد آخر ~30 طبقة من MobileNetV2 ونكمّل التدريب بمعدل تعلّم أقل بـ10 أضعاف،
عشان النموذج يتعلم تفاصيل أدق خاصة بصور الآفات الجلدية. هذا عادة بيرفع الـ Recall
للفئات الخطيرة (mel, bcc, akiec) بشكل ملحوظ.

In [ ]:
from tensorflow.keras.optimizers import Adam

# فك تجميد آخر 30 طبقة فقط من القاعدة (مش النموذج كامل)
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# إعادة تجميع بمعدل تعلّم أقل بكثير (مهم جداً - لو بقي عالي بيخرب الأوزان المدرَّبة)
model.compile(optimizer=Adam(learning_rate=1e-5),
              loss="categorical_crossentropy", metrics=["accuracy"])

fine_tune_callbacks = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    ModelCheckpoint(CHECKPOINT_PATH, monitor="val_accuracy", save_best_only=True),
]

history_ft = model.fit(
    train_gen, validation_data=val_gen, epochs=10,
    class_weight=class_weights, callbacks=fine_tune_callbacks,
)

## 12.6) إعادة التقييم بعد Fine-Tuning + مصفوفة الالتباس
قارني هذه النتيجة مع تقرير التقييم قبل Fine-Tuning لتشوفي التحسّن، خصوصاً بعمود Recall
لفئات mel وbcc وakiec.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

val_gen.reset()
preds = model.predict(val_gen)
y_pred = np.argmax(preds, axis=1)
y_true = val_gen.classes
class_names = list(class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45)
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
ax.set_xlabel("القيمة المتوقعة"); ax.set_ylabel("القيمة الفعلية")
ax.set_title("مصفوفة الالتباس (Confusion Matrix)")
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, cm[i,j], ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()

## 13) رسم منحنيات التدريب

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("الدقة (Accuracy)"); axes[0].legend()

axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("الخسارة (Loss)"); axes[1].legend()
plt.tight_layout(); plt.show()

## 14) النموذج محفوظ أصلاً على Drive - تنزيله لجهازك (اختياري)
بما إنه محفوظ أوتوماتيكياً على Drive بمسار `MyDrive/skin_cancer_project/best_model.h5`،
ممكن تنزّليه من موقع Google Drive مباشرة، أو تشغّلي هذه الخلية لتنزيله من Colab.

In [ ]:
from google.colab import files
files.download(CHECKPOINT_PATH)

---
# 15) معمارية محسّنة: EfficientNetB0 + Focal Loss

بعد ما شفنا إنه MobileNetV2 + class_weight وصل لسقف حوالي 61-62% Accuracy، وإنه Fine-Tuning
البسيط ما حسّن (وحتى أثّر سلباً على Recall لفئة mel)، رح نجرب معمارية أقوى:

**ليش EfficientNetB0؟**
- بنية أحدث وأكثر كفاءة من MobileNetV2 (نفس حجم تقريباً بس دقة أعلى على ImageNet وعلى أبحاث HAM10000 منشورة)
- بتلتقط تفاصيل نسيجية (texture) أدق، مهم لتمييز mel عن nv وbkl

**ليش Focal Loss بدل class_weight؟**
- class_weight بيخلي النموذج "يتحمّس" للفئات النادرة بشكل مبالغ فيه أحياناً (شفنا هيك: 184 حالة nv انصنّفت غلط كـ mel)
- Focal Loss بيركّز تلقائياً على الأمثلة الصعبة (اللي النموذج غير متأكد منها) بدل ما يعاقب كل الفئة الكبيرة بنفس القوة، وعادة بيعطي توازن أفضل بين Precision وRecall

## 15.1) مولّدات بيانات جديدة (بمعالجة مسبقة خاصة بـ EfficientNet)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

# EfficientNet له دالة معالجة مسبقة خاصة به (مختلفة عن rescale=1/255 العادية)
eff_train_datagen = ImageDataGenerator(
    preprocessing_function=eff_preprocess,
    rotation_range=30, width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True, vertical_flip=True, zoom_range=0.15,
    brightness_range=[0.85, 1.15], fill_mode="nearest",
)
eff_val_datagen = ImageDataGenerator(preprocessing_function=eff_preprocess)

eff_train_gen = eff_train_datagen.flow_from_dataframe(
    dataframe=train_df, directory=IMAGES_DIR, x_col="filename", y_col="dx",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical",
)
eff_val_gen = eff_val_datagen.flow_from_dataframe(
    dataframe=val_df, directory=IMAGES_DIR, x_col="filename", y_col="dx",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False,
)
print("الفئات:", eff_train_gen.class_indices)

## 15.2) تعريف Focal Loss

In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K

def categorical_focal_loss(gamma=2.0, alpha=0.25):
    """
    gamma: كل ما زاد، النموذج يركّز أكتر على الأمثلة الصعبة (اللي مش متأكد منها)
    alpha: وزن عام للتوازن بين الفئات
    """
    def focal_loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1.0 - K.epsilon())
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * tf.math.pow((1 - y_pred), gamma)
        loss = weight * cross_entropy
        return tf.reduce_sum(loss, axis=-1)
    return focal_loss

## 15.3) بناء نموذج EfficientNetB0 (أو استكمال من Drive إذا موجود)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam

EFF_CHECKPOINT_PATH = f"{PROJECT_DIR}/best_model_efficientnet.h5"

if os.path.exists(EFF_CHECKPOINT_PATH):
    print("✅ تم العثور على نموذج EfficientNet محفوظ سابقاً - جاري الاستكمال منه")
    eff_model = load_model(EFF_CHECKPOINT_PATH, compile=False)
    eff_base_model = eff_model.layers[1]  # للاستخدام لاحقاً بمرحلة فك التجميد
else:
    print("⏳ بناء نموذج EfficientNetB0 جديد")
    eff_base_model = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224,224,3))
    eff_base_model.trainable = False

    x = GlobalAveragePooling2D()(eff_base_model.output)
    x = BatchNormalization()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.4)(x)
    output = Dense(len(eff_train_gen.class_indices), activation="softmax")(x)
    eff_model = Model(inputs=eff_base_model.input, outputs=output)

eff_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=categorical_focal_loss(gamma=2.0, alpha=0.25),
    metrics=["accuracy"],
)
eff_model.summary()

## 15.4) المرحلة الأولى: تدريب الرأس فقط (القاعدة مجمّدة)
هاي أسرع مرحلة، بتدرّب بس الطبقات الجديدة اللي أضفناها فوق EfficientNet.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

phase1_callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ModelCheckpoint(EFF_CHECKPOINT_PATH, monitor="val_accuracy", save_best_only=True),
]

history_eff_p1 = eff_model.fit(
    eff_train_gen, validation_data=eff_val_gen, epochs=8,
    callbacks=phase1_callbacks,
)

## 15.5) المرحلة الثانية: فك تجميد القاعدة بالكامل + Fine-Tuning بمعدل تعلّم منخفض جداً
هاي المرحلة الأهم للوصول لدقة أعلى — بتاخذ وقت أطول لأنها بتدرّب النموذج كامل.

In [ ]:
eff_base_model.trainable = True

eff_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=categorical_focal_loss(gamma=2.0, alpha=0.25),
    metrics=["accuracy"],
)

phase2_callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(EFF_CHECKPOINT_PATH, monitor="val_accuracy", save_best_only=True),
]

history_eff_p2 = eff_model.fit(
    eff_train_gen, validation_data=eff_val_gen, epochs=15,
    callbacks=phase2_callbacks,
)

## 15.6) التقييم النهائي + مصفوفة الالتباس

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

eff_val_gen.reset()
preds = eff_model.predict(eff_val_gen)
y_pred = np.argmax(preds, axis=1)
y_true = eff_val_gen.classes
class_names = list(eff_train_gen.class_indices.keys())

print("=== تقرير EfficientNetB0 + Focal Loss ===")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45)
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
ax.set_xlabel("القيمة المتوقعة"); ax.set_ylabel("القيمة الفعلية")
ax.set_title("مصفوفة الالتباس - EfficientNetB0")
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, cm[i,j], ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()

## 15.7) تنزيل نموذج EfficientNet
محفوظ أصلاً على Drive بمسار `MyDrive/skin_cancer_project/best_model_efficientnet.h5`.

In [ ]:
from google.colab import files
files.download(EFF_CHECKPOINT_PATH)